# Milestone 1 Lab, notebook 3 of 6: Ask the LLM with all the data

## Where we are

| Notebook | What you do |
|---|---|
| 1. The business question | Read the brief, explore the data, and answer the question by hand |
| 2. Your first LLM calls | Call a model, get a structured reply, and have AI draft the metric definitions |
| **3. Ask the LLM with all the data** (you are here) | Put the data in the prompt, ask the question, and check the answer |
| 4. What is a tool, what is an agent | Build one tool, run the agent loop by hand, then let LangChain run it |
| 5. The agent answers the question | Give an agent six tools and compare it with the single call |
| 6. Verify and go to production | Check an answer with code, and see the production version |

Model requests in this notebook: **1**. OpenRouter's free models allow about 50 a day.

## What this notebook is for

Time to ask the leader's question. We take the simplest possible approach: paste the whole profit table into the prompt and ask.

Then we do what notebook 1 prepared us for: hold the model's answer against the answer key, line by line.

In [ ]:
import json

import pandas as pd
from lab_helpers import (
    LAST_MONTH,
    MAIN_QUESTION,
    PREVIOUS_MONTH,
    AnalystAnswer,
    check_figures,
    compare_with_key,
    connect,
    definitions,
    flags,
    load_result,
    products,
    profit,
    save_result,
    show_answer,
    show_doc,
)

client, MODEL = connect()

`connect()` is the connection code you wrote in notebook 2, now in `lab_helpers.py`.

## Step 1: Recall the correct answer

This is the answer key you built by hand in notebook 1. A correct answer must contain these facts.

In [ ]:
answer_key = load_result("answer_key", required_from="notebook 1")

print("Confidence should be:", answer_key["confidence"])
print("Data problems to mention:", answer_key["flags"])
pd.DataFrame(answer_key["figures"])[["what", "entity_id", "kind", "value"]]

## Step 2: Decide the shape of the answer

In notebook 2 we gave the model a small form to fill in. We do the same here with a bigger form, called `AnalystAnswer`.

The BRD says what a good answer contains. The form turns that list into fields.

In [ ]:
show_doc("BRD.md", "5. What a good answer contains")

| Field in the form | What it holds |
|---|---|
| `headline` | The answer in a sentence or two |
| `narrative` | A short explanation |
| `figures` | Every number, each labelled with its metric, month, and product or category |
| `definitions_used` | Which metric definitions the answer relies on |
| `caveats` | Data problems that affect the figures |
| `confidence` | High, medium, or low, with reasons |
| `sources` | The tables the figures came from |
| `status`, `decline_reason` | Whether the question was answered or declined, and why |

The important one is `figures`. Because every number sits in its own labelled slot, code can check each one. The form is defined in `lab_helpers.py`, and the full detail is in section 5 of `docs/LLD.md`.

In [ ]:
list(AnalystAnswer.model_fields)

## Step 3: Build the prompt

The system prompt has two parts. First, the instructions.

In [ ]:
INSTRUCTIONS = f"""You are the profit analyst for Kestrel Ridge Outfitters, a fictional online retailer of outdoor gear.
Answer the retail leader's question about profit.

- "Last month" is {LAST_MONTH}. The month before is {PREVIOUS_MONTH}. These are the only months with data.
- Put every number you use in figures, with its metric, month, and entity_id (a product_id such as P001, or a category name).
- For a change between the two months, use kind "change" and month {LAST_MONTH}.
- List every quality flag that affects your figures in caveats, with its flag_id.
- Set confidence to low if a high severity flag affects your figures, medium for a medium severity flag, otherwise high.
- A blank value is unavailable. Report it as null. Never treat it as zero or estimate it.
"""

Second, the data. The model can't open our files, so we paste the tables into the prompt as text. This is literally what the model will read:

In [ ]:
profit_as_text = profit.merge(products, on="product_id").to_csv(index=False)

print(profit_as_text[:600], "...")

In [ ]:
SYSTEM_PROMPT = f"""{INSTRUCTIONS}
# Metric definitions

{json.dumps(definitions["metrics"], indent=2)}

# Table: monthly_product_profit (one row per product per month)

{profit_as_text}

# Table: quality_flags

{flags.to_csv(index=False)}
"""

print(f"The system prompt is {len(SYSTEM_PROMPT.split()):,} words long")

Stop and think about what we are asking the model to do with that wall of text:

- to find the **top products**, it must sort 12 numbers
- to find the **top category**, it must add up four numbers, three times
- to say **what changed**, it must do that for both months and subtract
- and it must notice the **Camp trap** from notebook 1

A language model is built to predict text. It is not a calculator.

## Step 4: Ask

This is the same structured call as in notebook 2, with the bigger form.

In [ ]:
print(MAIN_QUESTION)

response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": MAIN_QUESTION}],
    response_format=AnalystAnswer,
    extra_body={"provider": {"require_parameters": True}},
)
llm_answer = response.choices[0].message.parsed

print(f"Answered by {response.model}")

> **If that cell failed**, run it again. A different free model will pick up the next request.

Here is the answer, displayed the way the leader would read it.

In [ ]:
show_answer(llm_answer)

It reads well. It is fluent and confident. **But is it right?** You cannot tell by reading it. A correct answer and a wrong answer look exactly the same.

## Step 5: Check it against the answer key

`compare_with_key` puts the correct answer beside what the model said, one line per fact.

- **yes**: the model gave this fact and got it right
- **NO**: the model gave it and got it wrong
- **not given**: the model left it out (or labelled it in a way we couldn't match)

In [ ]:
compare_with_key(answer_key, {"LLM": llm_answer})

Look especially at three lines:

- the **category total** (did it add four numbers correctly?)
- the **Camp change** (did it fall into the trap and report a fall?)
- the **confidence** (did it apply the rule?)

The table above only covers the facts in our key. The model may have given other numbers too. `check_figures` checks every number it gave against the truth.

In [ ]:
check_figures(llm_answer)

## What we learned

Your result depends on which free model answered. You may see several wrong numbers, or you may see none. Either way, this approach has three problems that don't go away:

1. **You can't tell if it's right without an answer key.** We only know because we did the work by hand first. For a new question, there is no key.
2. **You can't show where a number came from.** The model read a wall of text and produced a figure. There is nothing to point to.
3. **It doesn't scale.** We pasted 24 rows into the prompt. A real retailer has millions. They will not fit, and the more you paste, the more mistakes you get.

A profit figure that is confidently wrong is worse than no figure. Imagine the leader cutting Camp's budget because of a "fall" that was really one product's missing cost.

So here is the question for the next notebook: can we keep what the model is good at (understanding the question, explaining the result) and take the arithmetic away from it?

Save this answer so we can compare it with the agent's later.

In [ ]:
save_result("llm_answer", llm_answer.model_dump())

**Next: notebook 4**, where we meet tools and agents.